### **Clean-unload all aad modules**

In [1]:
# If the package is updated, run this cell then run the next cell.
# Clean-unload any previously imported aad modules from this kernel
# Clean-unload all aad modules and old names in this notebook
import sys, gc

# 1) Drop loaded modules
to_drop = [m for m in list(sys.modules) if m == "aad" or m.startswith("aad.")]
for m in to_drop:
    sys.modules.pop(m, None)

# 2) Drop any previously imported names from globals to avoid shadowing
for name in ["aad", "ADVar", "grad", "grads", "grads_list", "reverse"]:
    if name in globals():
        del globals()[name]

gc.collect()
print("Dropped modules:", to_drop if to_drop else "<none>")

Dropped modules: <none>


### **Reload aad**

In [7]:
!pip install numpy
import importlib, aad
import aad.core.var, aad.core.node, aad.core.tape, aad.core.engine, aad.core.seeds
import aad.ops.arithmetic, aad.ops.transcendental

# Reload core modules
importlib.reload(aad.core.var)
importlib.reload(aad.core.node)
importlib.reload(aad.core.tape)
importlib.reload(aad.core.engine)
importlib.reload(aad.core.seeds)

# Reload ops AFTER core so they bind the fresh ADVar
importlib.reload(aad.ops.arithmetic)
importlib.reload(aad.ops.transcendental)
importlib.reload(aad.ops.special)

# Re-import the fresh symbols
from aad.core.var import ADVar
from aad.core.seeds import grad, grads, grads_list
from aad.core.engine import reverse, zero_adjoints


[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


### **An easy test**

In [8]:
# ------------------ 1. Single-input grad ------------------ #
f = lambda x: x*x + 3*x
print("grad at x=2:", grad(f, 2))  # expect 7.0

# ------------------ 2. Multi-input grads (dict version) ------------------ #
def f_dict(vars):
    # f(x,y) = x^2 + 3y
    return vars["x"]*vars["x"] + 3*vars["y"]

inputs = {"x": 2.0, "y": 4.0}
print("grads dict:", grads(f_dict, inputs))  
# expect {"x": 4.0, "y": 3.0}

# ------------------ 3. Multi-input grads (list version) ------------------ #
f_list = lambda xs: xs[0]*xs[0] + 3*xs[1]
print("grads list:", grads_list(f_list, [2.0, 4.0]))  
# expect [4.0, 3.0]

grad at x=2: 7.0
grads dict: {'x': np.float64(4.0), 'y': np.float64(3.0)}
grads list: [np.float64(4.0), np.float64(3.0)]


### **AAD Greeks for European Call (Black-Scholes)**

In [9]:
import numpy as np
from aad.ops.transcendental import log, exp, sqrt
from aad.ops.special import norm_cdf
from aad.core.seeds import grads

def bs_call_price(S, K, r, sigma, T):
    # Wrap only if not ADVar; constants can stay plain floats.
    from aad.core.var import ADVar as _ADVar  # dynamic lookup to survive reloads
    S     = S     if isinstance(S, _ADVar) else _ADVar(S,     requires_grad=False)
    K     = K     if isinstance(K, _ADVar) else _ADVar(K,     requires_grad=False)
    r     = r     if isinstance(r, _ADVar) else _ADVar(r,     requires_grad=False)
    sigma = sigma if isinstance(sigma, _ADVar) else _ADVar(sigma, requires_grad=False)
    T     = T     if isinstance(T, _ADVar) else _ADVar(T,     requires_grad=False)

    sqrtT = sqrt(T)
    d1 = (log(S / K) + (r + 0.5 * sigma * sigma) * T) / (sigma * sqrtT)
    d2 = d1 - sigma * sqrtT
    disc = exp(-r * T)
    return S * norm_cdf(d1) - K * disc * norm_cdf(d2)

# Parameters
S0, K, r, sigma, T = 100.0, 100.0, 0.02, 0.2, 1.0

# One reverse pass: request gradients w.r.t. S and sigma only
def f(vars):
    return bs_call_price(vars["S"], K, r, vars["sigma"], T)

g = grads(f, {"S": S0, "sigma": sigma})

delta = g["S"]
vega  = g["sigma"]

print("Delta (AAD, one pass) =", delta)
print("Vega  (AAD, one pass) =", vega)

Delta (AAD, one pass) = 0.579259709439103
Vega  (AAD, one pass) = 39.104269397545586


### **Closed-form Greeks for European Call (Black-Scholes)**

In [10]:
# Closed-form Greeks for European Call (Black-Scholes)

import numpy as np
from math import erf, sqrt, log, exp

# standard normal CDF/PDF
def N(x):
    return 0.5 * (1.0 + erf(x / sqrt(2.0)))

def phi(x):
    return np.exp(-0.5 * x * x) / np.sqrt(2.0 * np.pi)

def bs_delta(S, K, r, sigma, T):
    d1 = (log(S / K) + (r + 0.5 * sigma * sigma) * T) / (sigma * sqrt(T))
    return N(d1)

def bs_vega(S, K, r, sigma, T):
    d1 = (log(S / K) + (r + 0.5 * sigma * sigma) * T) / (sigma * sqrt(T))
    return S * sqrt(T) * phi(d1)

# Inputs
S0, K, r, sigma, T = 100.0, 100.0, 0.02, 0.2, 1.0

# Closed-form results
delta_cf = bs_delta(S0, K, r, sigma, T)
vega_cf  = bs_vega(S0, K, r, sigma, T)

print("Closed-form Delta =", delta_cf)
print("Closed-form Vega  =", vega_cf)

Closed-form Delta = 0.579259709439103
Closed-form Vega  = 39.104269397545586
